In [148]:


import pandas as pd
import numpy as np


from db import get_connected

conn = get_connected()

prices = pd.read_sql("SELECT * FROM prices ORDER BY ticker,date", conn)
news= pd.read_sql("SELECT * FROM news ORDER BY ticker, published", conn)
conn.close()

C:\Users\amito\AppData\Local\Temp\ipykernel_5744\1880426003.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  prices = pd.read_sql("SELECT * FROM prices ORDER BY ticker,date", conn)
C:\Users\amito\AppData\Local\Temp\ipykernel_5744\1880426003.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  news= pd.read_sql("SELECT * FROM news ORDER BY ticker, published", conn)


In [149]:
prices.shape


(5984, 9)

In [150]:
news.shape

(1832, 7)

In [151]:
prices["daily_return"] = prices.groupby("ticker")["close"].pct_change()

In [152]:
prices[["ticker", "date", "close", "daily_return"]].head(10)

,ticker,date,close,daily_return
0,AAPL,2024-05-29,188.655777,NaN
1,AAPL,2024-05-30,189.647202,0.005255
2,AAPL,2024-05-31,190.598969,0.005019
3,AAPL,2024-06-03,192.363647,0.009259
4,AAPL,2024-06-04,192.680923,0.001649
5,AAPL,2024-06-05,194.187866,0.007821
6,AAPL,2024-06-06,192.809799,-0.007097
7,AAPL,2024-06-07,195.199097,0.012392
8,AAPL,2024-06-10,191.461456,-0.019148
9,AAPL,2024-06-11,205.370972,0.072649


In [153]:
prices["ma_7"]= prices.groupby("ticker")["close"].transform(lambda x: x.rolling(7).mean())

In [154]:
prices["ma_21"]= prices.groupby("ticker")["close"].transform(lambda x: x.rolling(21).mean())

In [155]:
prices["ma_ratio"] = prices["ma_7"] / prices["ma_21"]

In [156]:
prices[["ticker", "date", "close", "ma_7", "ma_21", "ma_ratio"]].head(25)

,ticker,date,close,ma_7,ma_21,ma_ratio
0,AAPL,2024-05-29,188.655777,NaN,NaN,NaN
1,AAPL,2024-05-30,189.647202,NaN,NaN,NaN
2,AAPL,2024-05-31,190.598969,NaN,NaN,NaN
3,AAPL,2024-06-03,192.363647,NaN,NaN,NaN
4,AAPL,2024-06-04,192.680923,NaN,NaN,NaN
5,AAPL,2024-06-05,194.187866,NaN,NaN,NaN
6,AAPL,2024-06-06,192.809799,191.563455,NaN,NaN
7,AAPL,2024-06-07,195.199097,192.498215,NaN,NaN
8,AAPL,2024-06-10,191.461456,192.757394,NaN,NaN
9,AAPL,2024-06-11,205.370972,194.867680,NaN,NaN


In [157]:
prices['daily_return'].head()

0         NaN
1    0.005255
2    0.005019
3    0.009259
4    0.001649
Name: daily_return, dtype: float64

In [158]:
prices['volume'].head()

0    53068000.0
1    49889100.0
2    75158300.0
3    50080500.0
4    47471400.0
Name: volume, dtype: float64

In [159]:
prices["volatility_7"] = prices.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(7).std())

In [160]:
prices["volume_change"] = prices.groupby("ticker")["volume"].pct_change()

In [161]:
def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [162]:
prices["rsi"] = prices.groupby("ticker")["close"].transform(calculate_rsi)

In [226]:
def calculate_macd(series):
    ema12 = series.ewm(span=12).mean()
    ema26 = series.ewm(span=26).mean()
    return ema12 - ema26

prices["macd"] = (
    prices.groupby("ticker")["close"]
    .transform(calculate_macd)
)

In [227]:
from ta.momentum import StochasticOscillator

def calculate_stochastic(group):
    return StochasticOscillator(
        high=group["high"],
        low=group["low"],
        close=group["close"],
        window=14
    ).stoch()

prices["stochastic"] = (
    prices.groupby("ticker", group_keys=False)
    .apply(calculate_stochastic)
)

In [228]:
prices[["ticker", "date", "close", "rsi"]].head(30)

,ticker,date,close,rsi
0,AAPL,2024-05-29,188.655777,NaN
1,AAPL,2024-05-30,189.647202,NaN
2,AAPL,2024-05-31,190.598969,NaN
3,AAPL,2024-06-03,192.363647,NaN
4,AAPL,2024-06-04,192.680923,NaN
5,AAPL,2024-06-05,194.187866,NaN
6,AAPL,2024-06-06,192.809799,NaN
7,AAPL,2024-06-07,195.199097,NaN
8,AAPL,2024-06-10,191.461456,NaN
9,AAPL,2024-06-11,205.370972,NaN


In [229]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

sentiment_model = pipeline("sentiment-analysis",
                           model="ProsusAI/finbert",
                           tokenizer="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [230]:
test_headlines = [
    "Apple beats earnings expectations, stock surges",
    "Tesla faces massive recall affecting 500,000 vehicles",
    "Microsoft reports steady quarterly revenue"
]

for headline in test_headlines:
    result = sentiment_model(headline)[0]
    print(f"{headline}")
    print(f"  → {result['label']}: {result['score']:.3f}\n")

Apple beats earnings expectations, stock surges
  → positive: 0.825

Tesla faces massive recall affecting 500,000 vehicles
  → negative: 0.953

Microsoft reports steady quarterly revenue
  → positive: 0.904



In [231]:
def get_sentiment_score(headline):
    try:
        result = sentiment_model(headline[:512])[0]  # FinBERT max 512 chars
        if result["label"] == "positive":
            return result["score"]
        elif result["label"] == "negative":
            return -result["score"]
        else:
            return 0.0
    except:
        return 0.0

In [232]:
print("Scoring headlines... this may take a minute")
news["sentiment"] = news["title"].apply(get_sentiment_score)

Scoring headlines... this may take a minute


In [233]:
news["date"] = news["published"].dt.date
daily_sentiment = news.groupby(["ticker", "date"])["sentiment"].mean().reset_index()
daily_sentiment.columns = ["ticker", "date", "sentiment_score"]

In [234]:
daily_sentiment.head()

,ticker,date,sentiment_score
0,AAPL,2026-05-27,0.077354
1,AAPL,2026-05-28,-0.054492
2,AAPL,2026-05-29,0.215910
3,AAPL,2026-06-01,0.024277
4,AAPL,2026-06-02,0.027405


In [235]:
prices["date"] = pd.to_datetime(prices["date"]).dt.date
features = prices.merge(daily_sentiment, on=["ticker", "date"], how="left")
features["sentiment_score"] = features["sentiment_score"].fillna(0)

In [236]:
features[["ticker", "date", "close", "daily_return", "ma_7", "ma_21", "ma_ratio", "rsi", "volatility_7", "volume_change", "sentiment_score"]].tail(10)

,ticker,date,close,daily_return,ma_7,ma_21,ma_ratio,rsi,volatility_7,volume_change,sentiment_score
5974,TSLA,2026-05-21,417.850006,0.001414,422.859994,405.199047,1.043586,59.741944,0.028760,-0.058678,0.000000
5975,TSLA,2026-05-22,426.010010,0.019529,420.108568,407.689048,1.030463,61.535812,0.027436,0.081333,0.000000
5976,TSLA,2026-05-26,433.589996,0.017793,418.721427,410.417143,1.020234,64.775463,0.028892,-0.009669,-0.432498
5977,TSLA,2026-05-27,440.359985,0.015614,421.309997,413.354761,1.019246,64.155045,0.021522,-0.021954,-0.064542
5978,TSLA,2026-05-28,442.100006,0.003951,425.897143,416.501428,1.022559,61.165550,0.015209,-0.276941,0.052558
5979,TSLA,2026-05-29,438.359985,-0.008460,430.790000,419.623333,1.026611,54.072077,0.013664,-0.055073,-0.497714
5980,TSLA,2026-06-01,415.880005,-0.051282,430.592856,421.254285,1.022168,38.690385,0.024700,0.472842,-0.281087
5981,TSLA,2026-06-02,423.739990,0.018900,431.434282,422.821904,1.020369,46.117544,0.025753,-0.163379,-0.250477
5982,TSLA,2026-06-03,423.700012,-0.000094,431.104283,424.307142,1.016019,40.478512,0.024606,0.178570,-0.512558
5983,TSLA,2026-06-04,418.600006,-0.012037,428.962856,425.699047,1.007667,39.390042,0.023464,-0.526008,0.000000


In [237]:
features["target"] = features.groupby("ticker")["close"].shift(-1) > features["close"]
features["target"] = features["target"].astype(int)

In [238]:
### How `shift(-1)` creates the target
"""
| date     | close  | shift(-1) (tomorrow's close) | tomorrow > today? | target |
|----------|--------|------------------------------|-------------------|--------|
| May 27   | 440.36 | 442.10                       | 442.10 > 440.36 ✓ | 1 (up) |
| May 28   | 442.10 | 438.36                       | 438.36 > 442.10 ✗ | 0 (down) |
| May 29   | 438.36 | NaN                          | no tomorrow yet   | dropped |

"""

"\n| date     | close  | shift(-1) (tomorrow's close) | tomorrow > today? | target |\n|----------|--------|------------------------------|-------------------|--------|\n| May 27   | 440.36 | 442.10                       | 442.10 > 440.36 ✓ | 1 (up) |\n| May 28   | 442.10 | 438.36                       | 438.36 > 442.10 ✗ | 0 (down) |\n| May 29   | 438.36 | NaN                          | no tomorrow yet   | dropped |\n\n"

In [239]:
feature_cols = ["daily_return", "ma_7", "ma_21", "ma_ratio", "rsi", "volatility_7", "volume_change", "sentiment_score","macd","stochastic"]
features_clean = features.dropna(subset=feature_cols + ["target"])
features_clean.head()

,id,ticker,date,open,high,low,close,volume,created_at,daily_return,ma_7,ma_21,ma_ratio,volatility_7,volume_change,rsi,macd,stochastic,sentiment_score,target
20,21,AAPL,2024-06-27,212.846236,213.887222,210.526336,212.261307,49772700.0,2026-05-29 23:16:11.616691,0.003986,209.049120,202.163531,1.034060,0.013660,-0.248296,70.012239,2.478507,78.253201,0.0,0
21,22,AAPL,2024-06-28,213.916953,214.214380,208.493929,208.811172,82542700.0,2026-05-29 23:16:11.616691,-0.016254,208.529338,203.123311,1.026615,0.014387,0.658393,63.705308,2.313652,65.846710,0.0,1
22,23,AAPL,2024-07-01,210.268521,215.641972,210.099983,214.888504,60402900.0,2026-05-29 23:16:11.616691,0.029104,209.530664,204.325278,1.025476,0.015815,-0.268222,72.526199,2.571253,87.015336,0.0,1
23,24,AAPL,2024-07-02,214.293699,218.487382,213.252728,218.378326,58046200.0,2026-05-29 23:16:11.616691,0.016240,211.340700,205.648105,1.027681,0.014689,-0.039016,65.641410,2.980891,99.202317,0.0,1
24,25,AAPL,2024-07-03,218.110647,219.647339,217.148976,219.647339,37369800.0,2026-05-29 23:16:11.616691,0.005811,213.239966,206.947328,1.030407,0.014555,-0.356206,61.367310,3.352594,100.000000,0.0,1


In [255]:
zero_count = (features_clean["sentiment_score"] == 0).sum()

total_count = len(features_clean)

print(f"Zeros: {zero_count}")
print(f"Total: {total_count}")
print(f"Percentage: {100 * zero_count / total_count:.2f}%")

Zeros: 5687
Total: 5784
Percentage: 98.32%


In [256]:
data=features_clean.sort_values(by='date')

dates = data['date'].unique()

cutoff_idx=int(len(dates)*0.8)

cutoff_date=dates[cutoff_idx]


In [257]:
train_df = data[data['date'] <= cutoff_date]
test_df  = data[data['date']  > cutoff_date]

In [258]:
train_df.shape

(4618, 20)

In [259]:
test_df.shape

(1166, 20)

In [260]:
train_df.tail()

,id,ticker,date,open,high,low,close,volume,created_at,daily_return,ma_7,ma_21,ma_ratio,volatility_7,volume_change,rsi,macd,stochastic,sentiment_score,target
406,407,AAPL,2026-01-12,258.679420,260.815436,256.323780,259.767395,4.526380e+07,2026-05-29 23:16:11.616691,0.003393,262.315495,269.574867,0.973071,0.007819,0.131680,19.829231,-3.523932,20.551143,0.0,1
5884,1913,TSLA,2026-01-12,441.230011,454.299988,438.000000,448.959991,6.164960e+07,2026-05-29 23:16:11.616691,0.008876,440.554286,461.210478,0.955213,0.025796,-0.084387,35.291962,-1.983015,33.024441,0.0,0
4641,909,MSFT,2026-01-12,474.556674,478.857498,473.571042,475.064392,2.351990e+07,2026-05-29 23:16:11.616691,-0.004382,475.360230,479.164715,0.992060,0.011990,0.271965,38.874681,-2.882379,38.019665,0.0,0
2573,4337,ETH-USD,2026-01-12,3118.828857,3166.219971,3068.072754,3092.325195,2.115271e+10,2026-05-29 23:16:11.616691,-0.008516,3134.832415,3050.916713,1.027505,0.020047,1.015078,62.198126,24.760249,45.449364,0.0,1
3123,1411,GOOGL,2026-01-12,325.570791,333.805015,324.771366,331.626526,3.392390e+07,2026-05-29 23:16:11.616691,0.010013,321.756356,313.535001,1.026221,0.009278,0.294104,88.023232,6.248395,92.414680,0.0,1


In [261]:
test_df.head()

,id,ticker,date,open,high,low,close,volume,created_at,daily_return,ma_7,ma_21,ma_ratio,volatility_7,volume_change,rsi,macd,stochastic,sentiment_score,target
4136,2416,JPM,2026-01-13,322.648751,325.195714,308.988680,309.316986,1.937120e+07,2026-05-29 23:16:11.616691,-0.041881,325.265381,320.670881,1.014328,0.022579,0.512772,40.431675,3.171209,1.304294,0.0,0
3124,1412,GOOGL,2026-01-13,334.714402,340.250483,333.385320,335.733673,3.351760e+07,2026-05-29 23:16:11.616691,0.012385,324.728551,314.655166,1.032014,0.009357,-0.011977,88.548579,7.044592,85.498908,0.0,0
4642,910,MSFT,2026-01-13,472.575483,473.670613,463.884208,468.583282,2.854580e+07,2026-05-29 23:16:11.616691,-0.013643,475.037384,478.557895,0.992644,0.009849,0.213687,34.092361,-3.424052,19.873698,0.0,0
1837,3607,BTC-USD,2026-01-13,91185.335938,96011.625000,90941.929688,95321.781250,5.498067e+10,2026-05-29 23:16:11.616691,0.045275,91511.023438,89987.873140,1.016926,0.021420,0.329761,71.911540,820.512153,92.232419,0.0,1
407,408,AAPL,2026-01-13,258.240222,261.324488,257.910847,260.565887,4.573080e+07,2026-05-29 23:16:11.616691,0.003074,260.895277,268.767795,0.970709,0.008541,0.010317,25.614566,-3.608845,24.164405,0.0,0


In [262]:
train_df["date"].max() < test_df["date"].min()

True

In [285]:
feature_cols = [
    "daily_return",
    "ma_7",
    "ma_21",
    "ma_ratio",
    "rsi",
    "volatility_7",
    "volume_change",
    "sentiment_score",
    "macd",
    "stochastic"
]



In [286]:
X_train=train_df[feature_cols]
y_train=train_df["target"]

In [287]:
X_test=test_df[feature_cols]
y_test=test_df["target"]

In [288]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

target
1    0.529233
0    0.470767
Name: proportion, dtype: float64
target
0    0.525729
1    0.474271
Name: proportion, dtype: float64


In [289]:
X_train.dtypes

daily_return       float64
ma_7               float64
ma_21              float64
ma_ratio           float64
rsi                float64
volatility_7       float64
volume_change      float64
sentiment_score    float64
macd               float64
stochastic         float64
dtype: object

In [290]:
import numpy as np

numeric_X_train = X_train.select_dtypes(include=["number"])

np.isinf(numeric_X_train).sum().sum()

np.int64(0)

In [291]:
numeric_X_train = X_test.select_dtypes(include=["number"])

np.isinf(numeric_X_train).sum().sum()

np.int64(0)

In [292]:
from xgboost import XGBClassifier

In [293]:
X_train.shape


(4618, 10)

In [294]:

X_test.shape

(1166, 10)

In [304]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(scale)

0.8895253682487725


In [310]:
model = XGBClassifier(
    n_estimators=300, # 100 trees
    max_depth =6,
    learning_rate = 0.05,
    scale_pos_weight=scale, #
    random_state = 42,
)

In [311]:
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [312]:
y_pred = model.predict(X_test)

In [313]:
from sklearn.metrics import accuracy_score,precision_score, f1_score,recall_score


In [314]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.48713550600343053
Precision: 0.46715328467153283
Recall: 0.5786618444846293
F1: 0.5169628432956381


In [315]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[248 365]
 [233 320]]


In [301]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
4,rsi,0.120585
2,ma_21,0.118812
0,daily_return,0.112655
9,stochastic,0.111070
5,volatility_7,0.110177
8,macd,0.107788
1,ma_7,0.106880
3,ma_ratio,0.106532
6,volume_change,0.105501
7,sentiment_score,0.000000


In [321]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("marketpulse_xgboost")

with mlflow.start_run():

    mlflow.log_param("n_estimators", 600)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred))
    mlflow.xgboost.log_model(model,"model_1")

print("Run logged successfully")

2026/06/05 00:04:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run logged successfully


In [323]:
print(mlflow.get_tracking_uri())
print(mlflow.get_experiment_by_name("marketpulse_xgboost"))

sqlite:///mlflow.db
<Experiment: artifact_location='file:C:/Users/amito/PycharmProjects/PythonProject/market-oracle/mlruns/1', creation_time=1780597083645, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780597083645, lifecycle_stage='active', name='marketpulse_xgboost', tags={}, trace_location=None, workspace='default'>
